# 14 — JNLPBA (biomedical) Arm 3: LLM prompting

LLM prompting (OpenAI `gpt-4o-mini`) on the biomedical domain, matching the Arm 3 setup of
Notebook 09: same prompt/parse/metric, same single demo seed (42), demonstrations drawn from the
few-shot splits created by Notebook 13.

**Cost cap:** JNLPBA's test set is large (~3,856 sentences), so we evaluate on a fixed,
randomly-sampled **800-sentence subset** (seed 42) to keep cost near $1. This is a documented
budget bound; `n_eval` is recorded in the results. Output goes to `results/jnlpba/llm_jnlpba.csv`
(a separate file — nothing existing is touched).

In [1]:
!pip -q install "seqeval==1.2.2" openai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Step 1 — Config + key

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, re, json, time, random
from pathlib import Path
import pandas as pd

PROCESSED      = Path('/content/drive/MyDrive/AAI590/data/processed')
FEWSHOT_SPLITS = PROCESSED / 'fewshot_splits'
JN_RESULTS     = PROCESSED / 'results' / 'jnlpba'
JN_RESULTS.mkdir(parents=True, exist_ok=True)

DS = 'jnlpba'
BUDGETS = [50, 100, 200]
DEMO_SEED = 42
EVAL_LIMIT = 800          # fixed cost cap on JNLPBA's large test set (set None for full test)
EVAL_SAMPLE_SEED = 42
OPENAI_MODEL = 'gpt-4o-mini'
MAX_OUTPUT_TOKENS = 512
PRICE_PER_MTOK = {'input': 0.15, 'output': 0.60, 'cache_read': 0.075}   # gpt-4o-mini

# OpenAI key: paste temporarily (BLANK before commit), else Colab secret, else hidden prompt
OPENAI_API_KEY = ""

if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
if not os.environ.get('OPENAI_API_KEY'):
    try:
        from google.colab import userdata
        if userdata.get('OPENAI_API_KEY'):
            os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    except Exception:
        pass
if not os.environ.get('OPENAI_API_KEY'):
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OPENAI_API_KEY (hidden): ')

RESULTS_CSV = JN_RESULTS / 'llm_jnlpba.csv'
PRED_DIR = JN_RESULTS / 'llm_predictions'; PRED_DIR.mkdir(exist_ok=True)

def load_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]

assert (FEWSHOT_SPLITS / DS).exists(), 'Run Notebook 13 first (jnlpba few-shot splits missing).'
print('ready | model:', OPENAI_MODEL, '| eval cap:', EVAL_LIMIT)

Mounted at /content/drive
ready | model: gpt-4o-mini | eval cap: 800


## Step 2 — Metrics + prompt/parse helpers (identical to Notebook 09)

In [3]:
from seqeval.metrics import classification_report
from seqeval.metrics.sequence_labeling import get_entities

def collapse(t): return 'O' if t == 'O' else ('B-ENT' if t.startswith('B-') else 'I-ENT')

def compute_entity_metrics(true_tags, pred_tags):
    rep = classification_report(true_tags, pred_tags, output_dict=True, zero_division=0)
    micro = rep['micro avg']
    per_type = {k: {'precision': float(v['precision']), 'recall': float(v['recall']),
                    'f1': float(v['f1-score']), 'support': int(v['support'])}
                for k, v in rep.items() if k not in ('micro avg', 'macro avg', 'weighted avg')}
    tb = [[collapse(t) for t in s] for s in true_tags]; pb = [[collapse(t) for t in s] for s in pred_tags]
    brep = classification_report(tb, pb, output_dict=True, zero_division=0)['micro avg']
    return {'typed_precision': float(micro['precision']), 'typed_recall': float(micro['recall']),
            'typed_f1': float(micro['f1-score']), 'boundary_f1': float(brep['f1-score']),
            'support': int(micro['support']), 'per_type': per_type}

TYPE_NOTES = {'jnlpba': ('protein, DNA, RNA, cell_line (a named cultured cell line, e.g. Jurkat), '
                         'cell_type (a kind of cell, e.g. T lymphocytes)')}
def entity_types_of(ds):
    tr = load_jsonl(PROCESSED / ds / f'{ds}_train.jsonl')
    return sorted({t.split('-', 1)[1] for r in tr for t in r['tags'] if t != 'O'})
def gold_entities(tokens, tags):
    return [{'text': ' '.join(tokens[s:e+1]), 'type': et} for et, s, e in get_entities(tags)]
def format_example(tokens, tags=None):
    line = 'Sentence: ' + ' '.join(tokens)
    return line + '\nEntities:' if tags is None else line + '\nEntities: ' + json.dumps(gold_entities(tokens, tags))
def build_system_prompt(ds, demo_rows):
    types = entity_types_of(ds)
    parts = ['You are a named entity recognition tagger.',
             f"Entity types for this domain: {', '.join(types)}.",
             f"Type hints: {TYPE_NOTES[ds]}.",
             'For the given sentence, list every entity as a JSON array of objects with keys '
             '"text" (the exact contiguous span, copied verbatim) and "type" (one of the types above). '
             'Preserve order of appearance. If there are no entities, answer []. Answer with ONLY the JSON array.']
    if demo_rows:
        parts += ['', f'Here are {len(demo_rows)} labeled examples from this domain:', '']
        parts += [format_example(r['tokens'], r['tags']) for r in demo_rows]
    return '\n'.join(parts)
def parse_entities(text):
    try:
        out = json.loads(text); return out if isinstance(out, list) else None
    except json.JSONDecodeError:
        pass
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if m:
        try:
            out = json.loads(m.group(0)); return out if isinstance(out, list) else None
        except json.JSONDecodeError:
            return None
    return None
def entities_to_bio(tokens, entities, allowed):
    tags = ['O'] * len(tokens); lower = [t.lower() for t in tokens]
    canon = {t.lower(): t for t in allowed}
    for ent in entities or []:
        if not isinstance(ent, dict) or 'text' not in ent or 'type' not in ent: continue
        et = canon.get(str(ent['type']).lower())
        if et is None: continue
        span = str(ent['text']).split()
        if not span: continue
        placed = False
        for exact in (True, False):
            hay = tokens if exact else lower
            needle = span if exact else [w.lower() for w in span]
            for i in range(len(tokens) - len(span) + 1):
                if hay[i:i+len(span)] == needle and all(t == 'O' for t in tags[i:i+len(span)]):
                    tags[i] = f'B-{et}'
                    for k in range(i+1, i+len(span)): tags[k] = f'I-{et}'
                    placed = True; break
            if placed: break
    return tags

In [4]:
import openai
from openai import OpenAI
client = OpenAI()

def _create_with_retry(**kw):
    delay = 1.0
    for _ in range(8):
        try:
            return client.chat.completions.create(**kw)
        except openai.RateLimitError as e:
            if 'insufficient_quota' in str(e): raise
            time.sleep(delay); delay = min(30.0, delay*2)
        except (openai.APITimeoutError, openai.APIConnectionError, openai.InternalServerError):
            time.sleep(delay); delay = min(30.0, delay*2)
    return client.chat.completions.create(**kw)

## Step 3 — Run the LLM grid on the capped JNLPBA test subset

In [5]:
# fixed evaluation subset (reproducible) so every budget scores on the same sentences
full_test = load_jsonl(PROCESSED / DS / f'{DS}_test.jsonl')
if EVAL_LIMIT and len(full_test) > EVAL_LIMIT:
    idx = sorted(random.Random(EVAL_SAMPLE_SEED).sample(range(len(full_test)), EVAL_LIMIT))
    eval_rows = [full_test[i] for i in idx]
else:
    eval_rows = full_test
allowed = entity_types_of(DS)
print(f'evaluating on {len(eval_rows)} of {len(full_test)} JNLPBA test sentences')

done = set()
if RESULTS_CSV.exists():
    done = set(pd.read_csv(RESULTS_CSV).budget)

for budget in BUDGETS:
    if budget in done:
        print('skipping budget', budget); continue
    print(f'\n=========== jnlpba budget={budget} ===========')
    demos = load_jsonl(FEWSHOT_SPLITS / DS / f'{DS}_train_{budget}_seed_{DEMO_SEED}.jsonl')
    system_prompt = build_system_prompt(DS, demos)
    pred_fp = PRED_DIR / f'openai_{DS}_{budget}.jsonl'
    cached = {r['id']: r for r in load_jsonl(pred_fp)} if pred_fp.exists() else {}

    t0 = time.time(); parse_fail = 0
    totals = {'input_tokens': 0, 'output_tokens': 0, 'cache_read_tokens': 0, 'cost_usd': 0.0}
    gold, pred = [], []
    for i, row in enumerate(eval_rows):
        if row['id'] in cached:
            rec = cached[row['id']]
        else:
            resp = _create_with_retry(model=OPENAI_MODEL, max_tokens=MAX_OUTPUT_TOKENS, temperature=0,
                    messages=[{'role': 'system', 'content': system_prompt},
                              {'role': 'user', 'content': format_example(row['tokens'])}])
            u = resp.usage
            det = getattr(u, 'prompt_tokens_details', None)
            crd = (getattr(det, 'cached_tokens', 0) or 0) if det else 0
            cost = ((u.prompt_tokens - crd) * PRICE_PER_MTOK['input'] + crd * PRICE_PER_MTOK['cache_read']
                    + u.completion_tokens * PRICE_PER_MTOK['output']) / 1e6
            raw = resp.choices[0].message.content or ''
            ents = parse_entities(raw)
            rec = {'id': row['id'], 'pred_tags': entities_to_bio(row['tokens'], ents, allowed),
                   'parse_failed': ents is None, 'input_tokens': u.prompt_tokens,
                   'output_tokens': u.completion_tokens, 'cache_read_tokens': crd, 'cost_usd': cost}
            with open(pred_fp, 'a') as f: f.write(json.dumps(rec, ensure_ascii=True) + '\n')
        parse_fail += int(rec['parse_failed'])
        for k in totals: totals[k] += rec.get(k, 0)
        gold.append(row['tags']); pred.append(rec['pred_tags'])
        if (i+1) % 100 == 0: print(f"  {i+1}/{len(eval_rows)}  (${totals['cost_usd']:.3f})")

    m = compute_entity_metrics(gold, pred)
    row_out = {'method': 'llm_prompting', 'dataset': DS, 'budget': budget, 'seed': DEMO_SEED,
               'provider': 'openai', 'model': OPENAI_MODEL,
               'test_typed_f1': m['typed_f1'], 'test_boundary_f1': m['boundary_f1'],
               'support': m['support'], 'n_eval': len(eval_rows), 'parse_failures': parse_fail,
               'wall_seconds': round(time.time()-t0, 1), 'cost_usd': round(totals['cost_usd'], 4),
               'per_type': json.dumps(m['per_type'])}
    pd.DataFrame([row_out]).to_csv(RESULTS_CSV, mode='a', index=False, header=not RESULTS_CSV.exists())
    print(f"  typed F1={m['typed_f1']:.3f}  boundary F1={m['boundary_f1']:.3f}  "
          f"cost=${totals['cost_usd']:.3f}  {parse_fail} parse failures")

print('\nsaved ->', RESULTS_CSV)

evaluating on 800 of 3856 JNLPBA test sentences

=========== jnlpba budget=50 ===========
  100/800  ($0.037)
  200/800  ($0.073)
  300/800  ($0.109)
  400/800  ($0.145)
  500/800  ($0.182)
  600/800  ($0.219)
  700/800  ($0.256)
  800/800  ($0.292)
  typed F1=0.571  boundary F1=0.599  cost=$0.292  0 parse failures

=========== jnlpba budget=100 ===========
  100/800  ($0.067)
  200/800  ($0.134)
  300/800  ($0.199)
  400/800  ($0.266)
  500/800  ($0.333)
  600/800  ($0.400)
  700/800  ($0.468)
  800/800  ($0.534)
  typed F1=0.573  boundary F1=0.600  cost=$0.534  0 parse failures

=========== jnlpba budget=200 ===========
  100/800  ($0.124)
  200/800  ($0.246)
  300/800  ($0.368)
  400/800  ($0.489)
  500/800  ($0.612)
  600/800  ($0.733)
  700/800  ($0.856)
  800/800  ($0.978)
  typed F1=0.578  boundary F1=0.607  cost=$0.978  0 parse failures

saved -> /content/drive/MyDrive/AAI590/data/processed/results/jnlpba/llm_jnlpba.csv


## Step 4 — JNLPBA: LLM vs fine-tuning (Arm 3 vs Arm 1)

In [6]:
llm = pd.read_csv(RESULTS_CSV)[['budget', 'test_typed_f1', 'cost_usd', 'parse_failures', 'n_eval']]
ft_fp = JN_RESULTS / 'fewshot_jnlpba.csv'
if ft_fp.exists():
    ft = pd.read_csv(ft_fp).groupby('budget')['test_typed_f1'].mean().reset_index()
    cmp = ft.rename(columns={'test_typed_f1': 'fine_tuning_f1'}).merge(
        llm.rename(columns={'test_typed_f1': 'llm_f1'})[['budget', 'llm_f1']], on='budget')
    cmp['llm_minus_ft'] = (cmp['llm_f1'] - cmp['fine_tuning_f1']).round(3)
    display(cmp.round(3))
else:
    print('fine-tuning results not found; run Notebook 13 for the comparison.')
    display(llm.round(3))

,budget,fine_tuning_f1,llm_f1,llm_minus_ft
0,50,0.324,0.571,0.247
1,100,0.397,0.573,0.176
2,200,0.515,0.578,0.063
